# Fase 4 — Monitoramento Multimodal em Saúde da Mulher

**FIAP Pós-Tech — Machine Learning Engineering**

Demonstra o pipeline multimodal (vídeo + áudio + fusão):

1. Detecção de instrumentos cirúrgicos em vídeo (YOLOv8)
2. Transcrição + triagem psicológica de áudio (Azure/Whisper/mock)
3. Fusão em LangGraph → risco combinado → alerta à equipe
4. Segurança, explainability e auditoria (herdadas da Fase 3)

> Roda offline (backend mock/demo). Com Azure (F0) e um `best.pt` treinado no
> Colab, os mesmos passos usam os backends reais.

In [1]:
import os, sys, json
sys.path.insert(0, os.path.abspath(".."))
os.environ.setdefault("LLM_BACKEND", "mock")
os.environ.setdefault("AUDIO_BACKEND", "auto")  # cai em mock offline

'auto'

## 1. Vídeo — treino demo do YOLOv8 (offline)

In [2]:
from src.multimodal.video import train_yolo
out = train_yolo.run(mode="demo", epochs=8)   # gera dataset sintético + treina
print("pesos:", out["weights"])
print("mAP50:", out.get("mAP50"), "| mAP50-95:", out.get("mAP50_95"))

[demo] gerando dataset sintético de instrumentos…


[demo] treinando YOLOv8n por 8 épocas (offline)…


Ultralytics 8.4.152 🚀 Python-3.12.3 torch-2.14.0+cu130 CPU (Intel Xeon Processor @ 2.10GHz)


engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/claude/f4/tech-challenge-fase-3-main/data/multimodal/video/demo_instrumentos/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=8, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=instrumentos, nbs=64, nms=None, opset=None, optimize=False, optimizer=

Overriding model.yaml nc=80 with nc=4


...

Model summary (fused): 72 layers, 3,006,428 parameters, 0 gradients, 8.1 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.5s/it 1.5s

                   all         30         64      0.966      0.587      0.944      0.835


Speed: 1.1ms preprocess, 45.3ms inference, 0.0ms loss, 1.3ms postprocess per image


pesos: /home/claude/f4/tech-challenge-fase-3-main/results/multimodal/video/best.pt
mAP50: 0.9442 | mAP50-95: 0.8346


### 1.1 Inferência + relatório especializado

In [3]:
from src.multimodal.video.detect import detectar_video
from src.multimodal.video.report import gerar_relatorio, formatar_texto
res = detectar_video("../data/multimodal/video/demo_cirurgia.mp4")
rel_v = gerar_relatorio(res, procedimento="histerectomia (demo)")
print(formatar_texto(rel_v))
print("\nVídeo anotado salvo em:", res.video_anotado)

RELATÓRIO AUTOMÁTICO — ANÁLISE DE VÍDEO CIRÚRGICO
Procedimento: histerectomia (demo)
Duração analisada: 4.0s
Instrumentos identificados: bisturi
Contagem por instrumento: {'bisturi': 6}
Nível de alerta: AMARELO
Anomalias/desvios:
  - Baixa cobertura temporal de instrumentos (12%) — possível obstrução do campo ou pausa não documentada.
Observação: apoio à decisão baseado em visão computacional; a conduta é da equipe cirúrgica responsável.

Vídeo anotado salvo em: /home/claude/f4/tech-challenge-fase-3-main/results/multimodal/video/demo_cirurgia_anotado.mp4


## 2. Áudio — transcrição + triagem psicológica

Cenário de triagem de violência doméstica (deve elevar o alerta a VERMELHO).

In [4]:
from src.multimodal.audio.transcribe import transcrever
from src.multimodal.audio.analyze import analisar
from src.multimodal.audio.report import gerar_relatorio as rel_audio, formatar_texto as txt_audio
t = transcrever("../data/multimodal/audio/consulta_triagem_violencia.wav")
print("backend de transcrição:", t.backend)
rel_a = rel_audio(t, analisar(t.texto))
print(txt_audio(rel_a))

backend de transcrição: mock
RELATÓRIO AUTOMÁTICO — ANÁLISE DE ÁUDIO DE CONSULTA
Fonte: ../data/multimodal/audio/consulta_triagem_violencia.wav
Transcrição via: mock | duração: 3.0s
Nível de alerta: VERMELHO
Sinais de triagem: ansiedade gestacional, depressão pós-parto, possível violência doméstica
  - ansiedade gestacional: termos ['medo'] (intensidade 0.33)
  - depressão pós-parto: termos ['sozinha'] (intensidade 0.33)
  - possível violência doméstica: termos ['me empurrou', 'tenho medo dele', 'nao posso sair', 'ele controla', 'grita comigo', 'com medo em casa'] (intensidade 1.0)
Observação: Triagem de apoio à decisão (não é diagnóstico). Requer avaliação e validação de profissional de saúde. Em caso de risco, acionar o protocolo de proteção e a rede de apoio.


## 3. Fusão multimodal (LangGraph)

Combina os relatórios de vídeo e áudio, decide o risco e emite o alerta.

In [5]:
from src.multimodal.fusion.multimodal_graph import FusaoMultimodal
fusao = FusaoMultimodal()
estado = fusao.executar(rel_v.to_dict(), rel_a.to_dict(), paciente_id="PAC-0007")
print("LangGraph usado:", fusao.app is not None)
print()
print(estado["resumo_final"])

LangGraph usado: True

Paciente PAC-0007 | risco combinado: VERMELHO

Vídeo: amarelo | Áudio: vermelho

Achados:
- [vídeo] Baixa cobertura temporal de instrumentos (12%) — possível obstrução do campo ou pausa não documentada.
- [áudio] ansiedade gestacional
- [áudio] depressão pós-parto
- [áudio] possível violência doméstica

🚨 ALERTA MULTIMODAL — paciente PAC-0007 | risco VERMELHO. Acionar equipe especializada. Achados: [vídeo] Baixa cobertura temporal de instrumentos (12%) — possível obstrução do campo ou pausa não documentada.; [áudio] ansiedade gestacional; [áudio] depressão pós-parto; [áudio] possível violência doméstica

Trilha: receber_video → receber_audio → fundir → emitir_alerta → consolidar

Apoio à decisão multimodal — validação humana obrigatória; conduta é da equipe especializada.


## 4. Auditoria (rastreabilidade da decisão)

In [6]:
from src.assistant.safety.audit_log import AuditLogger
ev = AuditLogger().ler_eventos()
print(f"{len(ev)} eventos auditados." if ev else "sem eventos ainda")
if ev:
    print(json.dumps(ev[-1], ensure_ascii=False, indent=2))

29 eventos auditados.
{
  "pergunta": "[fusão multimodal] paciente PAC-0007",
  "resposta": "🚨 ALERTA MULTIMODAL — paciente PAC-0007 | risco VERMELHO. Acionar equipe especializada. Achados: [vídeo] Baixa cobertura temporal de instrumentos (12%) — possível obstrução do campo ou pausa não documentada.; [áudio] ansiedade gestacional; [áudio] depressão pós-parto; [áudio] possível violência doméstica",
  "backend_llm": "multimodal",
  "paciente_id": null,
  "fontes": [
    "[vídeo] Baixa cobertura temporal de instrumentos (12%) — possível obstrução do campo ou pausa não documentada.",
    "[áudio] ansiedade gestacional",
    "[áudio] depressão pós-parto",
    "[áudio] possível violência doméstica"
  ],
  "guardrail_bloqueou": false,
  "guardrail_categorias": [],
  "fluxo_no": null,
  "timestamp": "2026-09-15T01:28:02.332404+00:00"
}


---
### Conclusão

O pipeline funde vídeo e áudio num fluxo de decisão LangGraph que herda os
guardrails e a auditoria da Fase 3. Trocando os backends (YOLO real, Azure F0,
Ollama), a mesma solução opera em produção sem alterar a fusão.